# Notebook 2 - Graph Analytics: The F1 Teammate Network

**Goal:** Build a network graph to analyze the historical relationships between Formula 1 drivers using PySpark and GraphFrames.

**Vertices (Nodes):** Drivers.  
**Edges (Relationships):** Two drivers are connected if they raced for the same Constructor in the same Year.

This notebook explores the graph using PageRank, Connected Components, and Breadth-First Search (BFS).

## 1. Environment Setup & Data Loading
We start by installing the required libraries, initializing Spark 3.5.1 with the GraphFrames package, and loading the clean analytical Gold tables generated by the data engineering pipeline.

In [ ]:
!pip uninstall -y pyspark
!pip install --no-cache-dir pyspark==3.5.1 graphframes

Found existing installation: pyspark 4.0.2
Uninstalling pyspark-4.0.2:
  Successfully uninstalled pyspark-4.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 195.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 182.6 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=e00cafea76d2c6ed401e82769bfe4b50c4d984ec5acc853ad26fe7cd4eb9fe6d
  Stored in directory: /tmp/pip-ephem-wheel-cache-mdt3ot2m/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from google.colab import drive

drive.mount('/content/drive')

spark = (
    SparkSession.builder
    .appName("F1 Notebook 2 - Graph Analytics")
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

DATASET_PATH = "/content/drive/MyDrive/f1_dataset"
GOLD_PATH = f"{DATASET_PATH}/lakehouse/gold"

fact_race_result = spark.read.parquet(f"{GOLD_PATH}/fact_race_result")
dim_driver = spark.read.parquet(f"{GOLD_PATH}/dim_driver")
dim_constructor = spark.read.parquet(f"{GOLD_PATH}/dim_constructor")

print("Spark 3.5 initialized and Gold tables loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 3.5 initialized and Gold tables loaded successfully!


## 2. Building the GraphFrame
GraphFrames requires two specific DataFrames: one for vertices (must contain an `id` column) and one for edges (must contain `src` and `dst` columns).

In [ ]:
from graphframes import GraphFrame

vertices = dim_driver.select(
    F.col("driverId").alias("id"),
    "driver_name",
    "nationality"
).distinct()

# We self-join the fact_race_result table to find drivers who raced for the same team in the same year.
edges_raw = (
    fact_race_result.alias("f1")
    .join(
        fact_race_result.alias("f2"),
        (F.col("f1.constructorId") == F.col("f2.constructorId")) &
        (F.col("f1.year") == F.col("f2.year")) &
        (F.col("f1.driverId") != F.col("f2.driverId")) # Exclude self-matches
    )
    .select(
        F.col("f1.driverId").alias("src"),
        F.col("f2.driverId").alias("dst"),
        F.col("f1.year").alias("year"),
        F.col("f1.constructorId").alias("constructorId")
    )
    .distinct()
)

# Aggregate the years they were teammates to create a "relationship weight"
edges = (
    edges_raw.groupBy("src", "dst", "constructorId")
    .agg(F.count("year").alias("years_as_teammates"))
)

f1_graph = GraphFrame(vertices, edges)

print(f"Total Drivers (Vertices): {f1_graph.vertices.count()}")
print(f"Total Teammate Relationships (Edges): {f1_graph.edges.count()}")

Total Drivers (Vertices): 861
Total Teammate Relationships (Edges): 16042


## 3. PageRank: Identifying Central Drivers
PageRank measures node importance based on the quantity and quality of links to it. A driver gets a high PageRank if they had many teammates, and those teammates also had many teammates. This helps us find the "anchor" drivers of F1 history.

In [ ]:
# Run PageRank
# We use resetProbability=0.15 and maxIter=10 as standard starting parameters
pagerank_results = f1_graph.pageRank(resetProbability=0.15, maxIter=10)

# Show the top 10 most "central" drivers
(
    pagerank_results.vertices
    .orderBy(F.desc("pagerank"))
    .select("id", "driver_name", "pagerank")
    .show(10, truncate=False)
)

+---+-------------------+------------------+
|id |driver_name        |pagerank          |
+---+-------------------+------------------+
|427|Maurice Trintignant|5.700948409615734 |
|475|Stirling Moss      |5.493255373108041 |
|356|Jack Brabham       |5.005245825922141 |
|501|Harry Schell       |4.546791226378888 |
|456|Roy Salvadori      |4.487356053313713 |
|418|Masten Gregory     |4.395893011655027 |
|347|Jo Bonnier         |4.156731707461079 |
|486|Jack Fairman       |3.5733196672116962|
|197|Jean-Pierre Jarier |3.4933620948684907|
|346|Jo Siffert         |3.4672151622179266|
+---+-------------------+------------------+
only showing top 10 rows



## 4. Connected Components: Network Segmentation
Connected Components identifies groups of nodes that are connected to each other. In F1, we want to see if the entire driver network is one interconnected web, or if there are isolated clusters of drivers (e.g., drivers from specific isolated eras).

In [ ]:
# Set checkpoint directory (Required for Connected Components to truncate logical plans)
spark.sparkContext.setCheckpointDir("/tmp/graphframes_checkpoints")

# Run Connected Components
cc_results = f1_graph.connectedComponents()

# Analyze the distribution of components
(
    cc_results.groupBy("component")
    .agg(F.count("id").alias("driver_count"))
    .orderBy(F.desc("driver_count"))
    .show(10)
)

+---------+------------+
|component|driver_count|
+---------+------------+
|        1|         826|
|      409|           3|
|      820|           2|
|      448|           2|
|      142|           2|
|      776|           2|
|      798|           2|
|      739|           2|
|      442|           1|
|      415|           1|
+---------+------------+
only showing top 10 rows



## 5. Breadth-First Search (BFS): Degrees of Separation
BFS explores the graph level by level to find the shortest path between nodes. Here, we calculate the "degrees of separation" between a modern champion (Max Verstappen) and a historical champion (Ayrton Senna) based strictly on chains of teammates.

In [ ]:
# Find the shortest path of teammates between Ayrton Senna and Max Verstappen
bfs_path = f1_graph.bfs(
    fromExpr="driver_name = 'Ayrton Senna'",
    toExpr="driver_name = 'Max Verstappen'",
    maxPathLength=10
)

# Display the path
bfs_path.show(truncate=False)

+------------------------------+---------------+------------------------------+--------------+--------------------------------+-----------------+-----------------------------------+----------------+----------------------------+
|from                          |e0             |v1                            |e1            |v2                              |e2               |v3                                 |e3              |to                          |
+------------------------------+---------------+------------------------------+--------------+--------------------------------+-----------------+-----------------------------------+----------------+----------------------------+
|{102, Ayrton Senna, Brazilian}|{102, 14, 3, 1}|{14, David Coulthard, British}|{14, 24, 9, 1}|{24, Vitantonio Liuzzi, Italian}|{24, 817, 164, 1}|{817, Daniel Ricciardo, Australian}|{817, 830, 9, 3}|{830, Max Verstappen, Dutch}|
+------------------------------+---------------+------------------------------+---------